In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/MyDrive/PhD/fedlbs_IWCMC/forecast-fedlbs')

In [1]:
import numpy as np
import pandas as pd
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from data import fetch_dataset
from util import move_sliding_window, num_params
from model import LSTMNet, GRUNet
from algorithm import fedlbs

pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, floatmode='fixed')

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('running on gpu')

else:
    device = torch.device("cpu")

running on gpu


# Parameters

In [3]:
window_size = 90 # Define window_size period and split inputs/labels\
batch_size = 1024
label_col_index = 0

# seq_len = 90  # (timestamps)
hidden_dim = 256
n_layers = 2
lr = 0.001
output_dim = 1

#fed train params
num_local_epochs = 1
max_rounds = 100 #nb of total rounds for training
num_clients_per_round = 10 #number of clients to participate in training

# Data Preperation

In [4]:
dataframes = fetch_dataset()

Floor10
Floor11
Floor12
Floor13
Floor14
Floor15
Floor17
Floor18
Floor19
Floor20
Floor21
Floor22
Floor3
Floor4
Floor5
Floor6
Floor8
Floor9
Floor10
Floor11
Floor12
Floor14
Floor15
Floor17
Floor19
Floor20
Floor21
Floor22
Floor3
Floor4
Floor5
Floor6
Floor7
Floor8
Floor9


In [5]:
n_clients = len(dataframes) #total number of clients to partition data into

In [6]:
for _, df in dataframes.items():
    print(df.columns)
    print(df.shape)

Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(72864, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(72864, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', '

# Build the training set

In [7]:
train_loader = []
test_loader = []
label_scalers = []
i= 0
for _, df in dataframes.items():
    inputs_cols_indices = range(0, df.shape[1])  # use (total_demand,Year,Month,Day,Hour,Minute) columns as features
    if i ==10:
        break
    #move the window
    inputs, labels = move_sliding_window(
        df.values,
        window_size,
        inputs_cols_indices=inputs_cols_indices,
        label_col_index=label_col_index
    )

    # Normalize the input data columns
    sc = MinMaxScaler()
    # Obtaining the scaler for the labels(usage data) so that output can be re-scaled to actual value during evaluation
    label_sc = MinMaxScaler()

    # Split data into train/test portions and combining all data into a single array
    test_portion = int(0.2 * len(inputs))

    train_x = sc.fit_transform(inputs[:-test_portion].reshape(-1, window_size * df.shape[1]))
    train_x = train_x.reshape(-1, window_size, df.shape[1])
    train_y = label_sc.fit_transform(labels[:-test_portion])

    test_x = sc.transform(inputs[-test_portion:].reshape(-1, window_size * df.shape[1]))
    test_x = test_x.reshape(-1, window_size, df.shape[1])
    test_y = label_sc.transform(labels[-test_portion:])

    # test_x.append(testx)
    # test_y.append(testy)
    label_scalers.append(label_sc)

    # pytorch data loaders
    train_data = TensorDataset(torch.from_numpy(train_x).to('cpu'), torch.from_numpy(train_y).to('cpu'))
    train_loader.append(DataLoader(train_data, batch_size=batch_size, drop_last=True))# Drop the last incomplete batch
    test_data = TensorDataset(torch.from_numpy(test_x).to('cpu'), torch.from_numpy(test_y).to('cpu'))
    test_loader.append(DataLoader(test_data, batch_size=batch_size))# Drop the last incomplete batch

    # release some memory
    del train_x, train_y
    i=i+1

input_dim = next(iter(train_loader[0]))[0].shape[2]  # 22

(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)
(20358, 90, 23) (20358, 1)
(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)
(20358, 90, 23) (20358, 1)
(72774, 90, 23) (72774, 1)
(40806, 90, 23) (40806, 1)


# LSTM model

In [8]:
lstm = LSTMNet(input_dim, hidden_dim, output_dim, n_layers)
model_type = 'LSTM'
print(lstm)
print(num_params(lstm))

LSTMNet(
  (lstm): LSTM(23, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (relu): ReLU()
)
814337


## S =3

In [9]:
%%time
S = 3
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [10  1  8 18 14 16  6  4  2  5]
round 0, starting client 1/10, id: 10
Epoch [1/1], Train Loss: 0.012510053979765096
round 0, starting client 2/10, id: 1
Epoch [1/1], Train Loss: 0.011449552970589142
round 0, starting client 3/10, id: 8
Epoch [1/1], Train Loss: 0.012600665853824468
round 0, starting client 4/10, id: 18
Epoch [1/1], Train Loss: 0.035321539640426634
round 0, starting client 5/10, id: 14
Epoch [1/1], Train Loss: 0.012102681162234927
round 0, starting client 6/10, id: 16
Epoch [1/1], Train Loss: 0.014930141923416938
round 0, starting client 7/10, id: 6
Epoch [1/1], Train Loss: 0.009970678004070315
round 0, starting client 8/10, id: 4
Epoch [1/1], Train Loss: 0.030693459510803222
round 0, starting client 9/10, id: 2
Epoch [1/1], Train Loss: 0.01177363536511881
round 0, starting client 10/10, id: 5
Epoch [1/1], Train Loss: 0.015230924615846766
calc smape: 16.901507676841504%
MAE: 7.5969278266711076
RMSE: 13.59781294892968
Average Loss:  0.0

In [10]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}_C{num_clients_per_round}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}_C{num_clients_per_round}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}_C{num_clients_per_round}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}_C{num_clients_per_round}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}_C{num_clients_per_round}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}_C{num_clients_per_round}.npy', rmse_lbs)

C:\Users\User\anaconda3\envs\fed\lib\site-packages\numpy\lib\npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)


## S = 5

In [11]:
%%time
S = 5
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [18 16 10  2 17 13 11  0  3  4]
round 0, starting client 1/10, id: 18
Epoch [1/1], Train Loss: 0.012418060536479711
round 0, starting client 2/10, id: 16
Epoch [1/1], Train Loss: 0.011400515722925773
round 0, starting client 3/10, id: 10
Epoch [1/1], Train Loss: 0.012539949575771713
round 0, starting client 4/10, id: 2
Epoch [1/1], Train Loss: 0.035387361422181136
round 0, starting client 5/10, id: 17
Epoch [1/1], Train Loss: 0.01213484687988447
round 0, starting client 6/10, id: 13
Epoch [1/1], Train Loss: 0.014879491134446908
round 0, starting client 7/10, id: 11
Epoch [1/1], Train Loss: 0.01003849352335757
round 0, starting client 8/10, id: 0
Epoch [1/1], Train Loss: 0.03077549953013659
round 0, starting client 9/10, id: 3
Epoch [1/1], Train Loss: 0.011977000638060937
round 0, starting client 10/10, id: 4
Epoch [1/1], Train Loss: 0.015570442402555095
calc smape: 17.054663096126273%
MAE: 7.620540202182074
RMSE: 13.606881857575841
Average Loss:  0.0

In [12]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}_C{num_clients_per_round}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}_C{num_clients_per_round}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}_C{num_clients_per_round}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}_C{num_clients_per_round}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}_C{num_clients_per_round}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}_C{num_clients_per_round}.npy', rmse_lbs)

C:\Users\User\anaconda3\envs\fed\lib\site-packages\numpy\lib\npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)


## S= 7

In [13]:
%%time
S = 7
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [15 13 12  5 11  2  8  6  3 17]
round 0, starting client 1/10, id: 15
Epoch [1/1], Train Loss: 0.012494144038230716
round 0, starting client 2/10, id: 13
Epoch [1/1], Train Loss: 0.011405142905589723
round 0, starting client 3/10, id: 12
Epoch [1/1], Train Loss: 0.012518132978584624
round 0, starting client 4/10, id: 5
Epoch [1/1], Train Loss: 0.03541753478348256
round 0, starting client 5/10, id: 11
Epoch [1/1], Train Loss: 0.012091507336923053
round 0, starting client 6/10, id: 2
Epoch [1/1], Train Loss: 0.014822496808067494
round 0, starting client 7/10, id: 8
Epoch [1/1], Train Loss: 0.010075595642514855
round 0, starting client 8/10, id: 6
Epoch [1/1], Train Loss: 0.030770504474639894
round 0, starting client 9/10, id: 3
Epoch [1/1], Train Loss: 0.011770214014437175
round 0, starting client 10/10, id: 17
Epoch [1/1], Train Loss: 0.015331384260207415
calc smape: 17.119160230328397%
MAE: 7.8029101863467245
RMSE: 14.120606541107723
Average Loss:  0

In [14]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}_C{num_clients_per_round}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}_C{num_clients_per_round}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}_C{num_clients_per_round}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}_C{num_clients_per_round}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}_C{num_clients_per_round}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}_C{num_clients_per_round}.npy', rmse_lbs)

C:\Users\User\anaconda3\envs\fed\lib\site-packages\numpy\lib\npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)


## S=10

In [15]:
%%time
S = 10
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [17  0 12  2  3 16  7 11 13 15]
round 0, starting client 1/10, id: 17
Epoch [1/1], Train Loss: 0.012551763614672896
round 0, starting client 2/10, id: 0
Epoch [1/1], Train Loss: 0.011433664987894842
round 0, starting client 3/10, id: 12
Epoch [1/1], Train Loss: 0.012577327115910267
round 0, starting client 4/10, id: 2
Epoch [1/1], Train Loss: 0.03536446982373794
round 0, starting client 5/10, id: 3
Epoch [1/1], Train Loss: 0.012248990587457745
round 0, starting client 6/10, id: 16
Epoch [1/1], Train Loss: 0.014919984092037862
round 0, starting client 7/10, id: 7
Epoch [1/1], Train Loss: 0.0100556457764469
round 0, starting client 8/10, id: 11
Epoch [1/1], Train Loss: 0.03065350962181886
round 0, starting client 9/10, id: 13
Epoch [1/1], Train Loss: 0.012110887631674167
round 0, starting client 10/10, id: 15
Epoch [1/1], Train Loss: 0.014249991747595728
calc smape: 16.875794165932387%
MAE: 7.613815942915075
RMSE: 13.634882680577498
Average Loss:  0.00

In [16]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}_C{num_clients_per_round}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}_C{num_clients_per_round}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}_C{num_clients_per_round}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}_C{num_clients_per_round}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}_C{num_clients_per_round}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}_C{num_clients_per_round}.npy', rmse_lbs)

C:\Users\User\anaconda3\envs\fed\lib\site-packages\numpy\lib\npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)
